In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import duckdb

input_file = "/content/drive/MyDrive/meta_Home_and_Kitchen.jsonl"
output_file = "/content/drive/MyDrive/meta_Home_and_Kitchen_duckdb.parquet"

print("Đang tạo lại file Parquet bằng DuckDB...")

# read_json_auto sẽ tự động rà soát và thống nhất schema an toàn hơn
query = f"""
COPY (
    SELECT * FROM read_json_auto('{input_file}', ignore_errors=true)
) TO '{output_file}' (FORMAT PARQUET);
"""
duckdb.sql(query)

print("Hoàn tất! Hãy thử đọc lại file mới này bằng Polars.")

Đang tạo lại file Parquet bằng DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Hoàn tất! Hãy thử đọc lại file mới này bằng Polars.


In [ ]:
import duckdb

# Truy vấn trực tiếp vào file Parquet 3.2GB
# Hàm .df() sẽ chuyển kết quả về dạng Pandas DataFrame quen thuộc để dễ hiển thị trên Colab
query = """
SELECT * FROM '/content/drive/MyDrive/meta_Home_and_Kitchen.parquet'
LIMIT 5
"""
df = duckdb.sql(query).df()

print("Đọc thành công! Hiển thị 5 dòng đầu tiên:")
display(df)

Đọc thành công! Hiển thị 5 dòng đầu tiên:


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Amazon Home,Set of 4 Irish Coffee Glass Mugs Footed 10.5 o...,4.6,18,[☕PERFECT IRISH COFFEE MUG: With our clear gla...,[Set of 12 Footed 10.5 oz. Irish coffee mug th...,24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Irish Coffee Glass Coffee Mugs Reg...,LavoHome,"[Home & Kitchen, Kitchen & Dining, Dining & En...","{'Brand': '""LavoHome""', 'Material': '""Glass""',...",B07R3DYMH6,None
1,Amazon Home,Foaming Soap Dispenser Thick Ceramic Foam Hand...,4.4,135,[Saving money: You can DIY foam soap which wil...,[],24.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Foaming Soap Dispenser Ceramic Foa...,rejomiik,"[Home & Kitchen, Bath]","{'Package Dimensions': '""7.32 x 6.14 x 3.94 in...",B0BNZ8Q7YT,None
2,Amazon Home,Tapestry Trading 558W90 90 in. European Lace T...,5.0,3,"[Polyester,lace, European Lace Tablecloth, 100...",[Features. European Lace Tablecloth. 100 Polye...,45.64,[{'thumb': 'https://m.media-amazon.com/images/...,[],Tapestry Trading,"[Home & Kitchen, Kitchen & Dining, Kitchen & T...","{'Brand': '""Tapestry Trading""', 'Color': '""Whi...",B01508WQC6,None
3,Amazon Home,jersey seating 2 x Vinyl Air Lift Adjustable S...,4.3,167,"[Sleek chrome metal base, seat covered in Red ...",[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Small and Stylish Barstools ', 'ur...",jersey seating®,"[Home & Kitchen, Furniture, Game & Recreation ...","{'Color': '""Red""', 'Frame Material': '""Metal""'...",B00KKU8HTG,None
4,Amazon Home,Chisander 20 Inches Grey with White Super Soft...,4.6,67,[High-Quality Material: Made of high quality s...,[],9.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Chisander,"[Home & Kitchen, Seasonal Décor, Stockings & H...","{'Package Dimensions': '""9.65 x 5.43 x 1.85 in...",B0B61RJ848,None


In [ ]:
import duckdb
import subprocess

# Đảm bảo đường dẫn này khớp với file của bạn
jsonl_file = "/content/drive/MyDrive/meta_Home_and_Kitchen.jsonl"
parquet_file = "/content/drive/MyDrive/meta_Home_and_Kitchen.parquet"

print("⏳ Đang đếm số dòng trong file JSONL gốc (11GB)...")
print("(Quá trình này có thể mất vài phút)")

# Đếm dòng JSONL
try:
    result = subprocess.run(['wc', '-l', jsonl_file], capture_output=True, text=True, check=True)
    json_lines = int(result.stdout.split()[0])
    print(f"👉 Tổng số dòng file JSONL: {json_lines:,}")
except Exception as e:
    print(f"Lỗi khi đếm file JSONL: {e}")
    json_lines = 0

print("\n⏳ Đang đếm số dòng trong file Parquet (3.2GB)...")
# Đếm dòng Parquet bằng DuckDB
try:
    parquet_lines = duckdb.sql(f"SELECT COUNT(*) FROM '{parquet_file}'").fetchone()[0]
    print(f"👉 Tổng số dòng file Parquet: {parquet_lines:,}")
except Exception as e:
    print(f"Lỗi khi đếm file Parquet: {e}")
    parquet_lines = 0

print("\n" + "="*40)
print("🎯 KẾT QUẢ KIỂM TRA:")
if json_lines > 0 and parquet_lines > 0:
    if json_lines == parquet_lines:
        print("✅ Tuyệt vời! Số dòng khớp nhau 100%.")
    else:
        diff = json_lines - parquet_lines
        print(f"⚠️ Có sự chênh lệch: File Parquet ít hơn {diff:,} dòng so với file gốc.")
        print("💡 Giải thích: DuckDB đã tự động bỏ qua các dòng JSON bị lỗi cấu trúc.")

⏳ Đang đếm số dòng trong file JSONL gốc (11GB)...
(Quá trình này có thể mất vài phút)
👉 Tổng số dòng file JSONL: 3,735,584

⏳ Đang đếm số dòng trong file Parquet (3.2GB)...
👉 Tổng số dòng file Parquet: 3,735,584

🎯 KẾT QUẢ KIỂM TRA:
✅ Tuyệt vời! Số dòng khớp nhau 100%.


In [ ]:
import duckdb
import pandas as pd
from IPython.display import display

# Cài đặt tùy chọn hiển thị Pandas để dễ nhìn hơn trên Colab
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

parquet_file = "/content/drive/MyDrive/meta_Home_and_Kitchen.parquet"

print("Đang tiến hành phân tích EDA trực tiếp trên file 3.2GB...\n")
print("="*60)

# ---------------------------------------------------------
# 1. TỔNG SỐ LƯỢNG SẢN PHẨM (TOTAL RECORDS)
# ---------------------------------------------------------
total_records = duckdb.sql(f"SELECT COUNT(*) FROM '{parquet_file}'").fetchone()[0]
print(f"1. Tổng số lượng sản phẩm (Total Records): {total_records:,} sản phẩm\n")


# ---------------------------------------------------------
# 2. SỐ LƯỢNG CỘT VÀ Ý NGHĨA (SCHEMA)
# ---------------------------------------------------------
print("2. Cấu trúc dữ liệu (Schema):")
schema_df = duckdb.sql(f"DESCRIBE SELECT * FROM '{parquet_file}'").df()
# Chỉ lấy tên cột và kiểu dữ liệu để hiển thị cho gọn
display(schema_df[['column_name', 'column_type']])
print(f"-> Tổng số cột: {len(schema_df)} cột\n")


# ---------------------------------------------------------
# 3. PHÂN BỔ SẢN PHẨM THEO CATEGORY (TOP 10)
# ---------------------------------------------------------
# Lưu ý: Bộ dữ liệu Amazon 2023 thường lưu danh mục chính ở cột 'main_category'
print("3. Top 10 Danh mục (Categories) phổ biến nhất:")
try:
    top_categories = duckdb.sql(f"""
        SELECT main_category, COUNT(*) as count
        FROM '{parquet_file}'
        GROUP BY main_category
        ORDER BY count DESC
        LIMIT 10
    """).df()
    display(top_categories)
except Exception as e:
    print(f"⚠️ Có lỗi khi đếm category (Có thể tên cột bị khác): {e}")


# ---------------------------------------------------------
# 4. TỈ LỆ DỮ LIỆU BỊ THIẾU (MISSING VALUES)
# ---------------------------------------------------------
print("\n4. Tỉ lệ dữ liệu bị thiếu (Missing Values):")
# Cách tính: (Tổng số dòng - Số dòng có dữ liệu) / Tổng số dòng * 100
missing_query = f"""
    SELECT
        (1 - COUNT(brand) * 1.0 / {total_records}) * 100 AS brand_missing_pct,
        (1 - COUNT(title) * 1.0 / {total_records}) * 100 AS title_missing_pct,
        (1 - COUNT(description) * 1.0 / {total_records}) * 100 AS description_missing_pct
    FROM '{parquet_file}'
"""
try:
    missing_df = duckdb.sql(missing_query).df()
    print(f"   - Brand (Thương hiệu): {missing_df['brand_missing_pct'][0]:.2f}% bị thiếu")
    print(f"   - Title (Tiêu đề):     {missing_df['title_missing_pct'][0]:.2f}% bị thiếu")
    print(f"   - Description (Mô tả): {missing_df['description_missing_pct'][0]:.2f}% bị thiếu")
except Exception as e:
    print(f"⚠️ Lỗi khi tính missing value (Có thể một số cột không tồn tại): {e}")

print("\n" + "="*60)
print("✅ QUÁ TRÌNH EDA HOÀN TẤT!")

Đang tiến hành phân tích EDA trực tiếp trên file 3.2GB...

1. Tổng số lượng sản phẩm (Total Records): 3,735,584 sản phẩm

2. Cấu trúc dữ liệu (Schema):


,column_name,column_type
0,main_category,VARCHAR
1,title,VARCHAR
2,average_rating,DOUBLE
3,rating_number,BIGINT
4,features,VARCHAR[]
5,description,VARCHAR[]
6,price,DOUBLE
7,images,"STRUCT(thumb VARCHAR, ""large"" VARCHAR, variant..."
8,videos,"STRUCT(title VARCHAR, url VARCHAR, user_id VAR..."
9,store,VARCHAR


-> Tổng số cột: 14 cột

3. Top 10 Danh mục (Categories) phổ biến nhất:


,main_category,count
0,Amazon Home,2914556
1,None,363310
2,Tools & Home Improvement,168402
3,Toys & Games,52371
4,Industrial & Scientific,36016
5,Health & Personal Care,31922
6,AMAZON FASHION,28291
7,Office Products,27624
8,Sports & Outdoors,16632
9,Appliances,15534



4. Tỉ lệ dữ liệu bị thiếu (Missing Values):
⚠️ Lỗi khi tính missing value (Có thể một số cột không tồn tại): Binder Error: Referenced column "brand" not found in FROM clause!
Candidate bindings: "rating_number", "main_category", "parent_asin"

✅ QUÁ TRÌNH EDA HOÀN TẤT!


In [ ]:
import polars as pl

# Đường dẫn file
input_parquet = "/content/drive/MyDrive/meta_Home_and_Kitchen.parquet"
output_parquet = "/content/drive/MyDrive/meta_Home_and_Kitchen_cleaned.parquet"

print("⏳ Đang thiết lập Pipeline xử lý dữ liệu...")

# Dùng scan_parquet để đọc lười (LazyFrame) - Giúp xử lý file 3.2GB không tốn RAM
lf = pl.scan_parquet(input_parquet)

# ---------------------------------------------------------
# 1. LỌC DỮ LIỆU: Bỏ sản phẩm thiếu title hoặc thiếu định danh (parent_asin)
# ---------------------------------------------------------
lf_filtered = lf.drop_nulls(subset=["title", "parent_asin"])

# ---------------------------------------------------------
# 2. LÀM SẠCH VĂN BẢN & TOKENIZATION
# ---------------------------------------------------------
lf_cleaned = (
    lf_filtered
    .with_columns(
        # Bước A: Chuyển về chữ thường (lowercase)
        pl.col("title").str.to_lowercase().alias("title_clean")
    )
    .with_columns(
        # Bước B: Xóa HTML tags (ví dụ: <br>, <div>)
        pl.col("title_clean").str.replace_all(r"<[^>]+>", " ")
    )
    .with_columns(
        # Bước C: Xóa ký tự đặc biệt (Chỉ giữ lại a-z, số 0-9 và dấu cách)
        pl.col("title_clean").str.replace_all(r"[^a-z0-9\s]", " ")
    )
    .with_columns(
        # Gom các khoảng trắng thừa thành 1 dấu cách duy nhất và cắt khoảng trắng 2 đầu
        pl.col("title_clean").str.replace_all(r"\s+", " ").str.strip_chars()
    )
    .with_columns(
        # Bước D: Tokenization (Tách từ thành danh sách các tokens)
        pl.col("title_clean").str.split(" ").alias("title_tokens")
    )
)

# ---------------------------------------------------------
# 3. KẾT QUẢ SO SÁNH (Hiển thị 10 dòng thô vs đã xử lý)
# ---------------------------------------------------------
print("⏳ Đang trích xuất 10 dòng đầu tiên để so sánh...\n")
# .collect() ở đây chỉ yêu cầu Polars chạy trên 10 dòng đầu, trả kết quả ngay lập tức
comparison_df = lf_cleaned.select(["title", "title_clean", "title_tokens"]).head(10).collect()

print("="*80)
print("🎯 SO SÁNH TRƯỚC VÀ SAU KHI LÀM SẠCH (10 dòng đầu):")
print("="*80)

# Duyệt qua từng dòng để in ra màn hình cho dễ nhìn
for i, row in enumerate(comparison_df.iter_rows(named=True)):
    print(f"\n📦 SẢN PHẨM {i+1}:")
    # Cắt ngắn text ở 100 ký tự để màn hình không bị trôi
    print(f"🔸 Gốc (Raw)      : {row['title'][:100]}...")
    print(f"🔹 Đã làm sạch    : {row['title_clean'][:100]}...")
    print(f"🧩 Tokens (5 từ)  : {row['title_tokens'][:5]}...")

# ---------------------------------------------------------
# 4. LƯU DỮ LIỆU ĐÃ XỬ LÝ (Uncomment 3 dòng dưới để chạy thật)
# ---------------------------------------------------------
# print("\n" + "="*80)
# print("⏳ Đang chạy toàn bộ pipeline và lưu ra file Parquet mới...")
# lf_cleaned.sink_parquet(output_parquet)
# print(f"✅ Đã lưu thành công dữ liệu sạch tại: {output_parquet}")

⏳ Đang thiết lập Pipeline xử lý dữ liệu...
⏳ Đang trích xuất 10 dòng đầu tiên để so sánh...

🎯 SO SÁNH TRƯỚC VÀ SAU KHI LÀM SẠCH (10 dòng đầu):

📦 SẢN PHẨM 1:
🔸 Gốc (Raw)      : Set of 4 Irish Coffee Glass Mugs Footed 10.5 oz.Thick Wall Glass For Coffee, tea, Cappuccinos, Mulle...
🔹 Đã làm sạch    : set of 4 irish coffee glass mugs footed 10 5 oz thick wall glass for coffee tea cappuccinos mulled c...
🧩 Tokens (5 từ)  : ['set', 'of', '4', 'irish', 'coffee']...

📦 SẢN PHẨM 2:
🔸 Gốc (Raw)      : Foaming Soap Dispenser Thick Ceramic Foam Hand Soap Dispenser for Bathroom or Kitchen Sink, Liquid P...
🔹 Đã làm sạch    : foaming soap dispenser thick ceramic foam hand soap dispenser for bathroom or kitchen sink liquid pu...
🧩 Tokens (5 từ)  : ['foaming', 'soap', 'dispenser', 'thick', 'ceramic']...

📦 SẢN PHẨM 3:
🔸 Gốc (Raw)      : Tapestry Trading 558W90 90 in. European Lace Table Cloth, White...
🔹 Đã làm sạch    : tapestry trading 558w90 90 in european lace table cloth white...
🧩 Tokens (5 từ

In [ ]:
import duckdb
import pandas as pd

parquet_file = "/content/drive/MyDrive/meta_Home_and_Kitchen.parquet"

print("⏳ Đang kết nối DuckDB để tính toán Missing Values...\n")

try:
    # 1. Lấy danh sách toàn bộ các cột hiện có trong file Parquet
    columns_info = duckdb.sql(f"DESCRIBE SELECT * FROM '{parquet_file}'").fetchall()
    columns = [col[0] for col in columns_info]

    # 2. Lấy tổng số dòng
    total_rows = duckdb.sql(f"SELECT COUNT(*) FROM '{parquet_file}'").fetchone()[0]
    print(f"Tổng số dòng trong file: {total_rows:,}\n")

    # 3. Tạo câu lệnh SQL tự động đếm giá trị NULL cho TẤT CẢ các cột
    # Dùng CASE WHEN ... IS NULL để đảm bảo an toàn tuyệt đối với mọi kiểu dữ liệu (kể cả List/Struct)
    select_clauses = []
    for col in columns:
        select_clauses.append(f"SUM(CASE WHEN \"{col}\" IS NULL THEN 1 ELSE 0 END) * 100.0 / {total_rows} AS \"{col}\"")

    query = f"SELECT {', '.join(select_clauses)} FROM '{parquet_file}'"

    # 4. Thực thi truy vấn và in kết quả
    missing_df = duckdb.sql(query).df()

    print("-" * 55)
    print(f"{'TÊN CỘT ĐANG CÓ':<35} | {'TỶ LỆ THIẾU (%)'}")
    print("-" * 55)

    for col in columns:
        missing_pct = missing_df[col][0]
        print(f"{col:<35} | {missing_pct:.2f}%")

    print("-" * 55)

except Exception as e:
    print(f"⚠️ Đã xảy ra lỗi: {e}")

⏳ Đang kết nối DuckDB để tính toán Missing Values...

Tổng số dòng trong file: 3,735,584



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

-------------------------------------------------------
TÊN CỘT ĐANG CÓ                     | TỶ LỆ THIẾU (%)
-------------------------------------------------------
main_category                       | 9.73%
title                               | 0.00%
average_rating                      | 0.00%
rating_number                       | 0.00%
features                            | 0.00%
description                         | 0.00%
price                               | 65.76%
images                              | 0.00%
videos                              | 0.00%
store                               | 2.24%
categories                          | 0.00%
details                             | 0.00%
parent_asin                         | 0.00%
bought_together                     | 100.00%
-------------------------------------------------------
